In [ ]:

import re
import pandas as pd
import numpy as np

file = "spellcheck1.txt"
try:
    df = pd.read_csv(file, sep=None, engine="python")
except:
    df = pd.read_csv(file, sep=r'\s+', header=None, names=["incorrect", "correct"])

print("Dataset loaded successfully!")

print("\nFirst 10 rows:")
print(df.head(10))

print("\nColumns:")
print(df.columns)
if len(df.columns) >= 2:

    incorrect_column = df.columns[0]
    correct_column = df.columns[1]

else:
    raise ValueError("Dataset must contain incorrect and correct spelling columns.")

print("\nIncorrect column:", incorrect_column)
print("Correct column:", correct_column)

def clean_text(text):

    text = str(text).lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Clean both columns
df[incorrect_column] = df[incorrect_column].apply(clean_text)
df[correct_column] = df[correct_column].apply(clean_text)


print("\nCleaned Dataset:")
print(df.head(10))


# --------------------------------------------
# 4. REMOVE EMPTY VALUES
# --------------------------------------------

df = df[
    (df[incorrect_column] != "") &
    (df[correct_column] != "")
]

df = df.dropna()

print("\nDataset size:", len(df))


# --------------------------------------------
# 5. CREATE SPELLING DICTIONARY
# --------------------------------------------

spelling_dict = dict(
    zip(
        df[incorrect_column],
        df[correct_column]
    )
)

print("\nSpelling dictionary created.")


# --------------------------------------------
# 6. CREATE CORRECT WORD VOCABULARY
# --------------------------------------------

correct_words = list(
    df[correct_column].unique()
)

print("Number of correct words:", len(correct_words))


# --------------------------------------------
# 7. LEVENSHTEIN EDIT DISTANCE
# --------------------------------------------

def edit_distance(word1, word2):

    m = len(word1)
    n = len(word2)

    # NumPy matrix
    dp = np.zeros(
        (m + 1, n + 1),
        dtype=int
    )

    # Initialize first column
    for i in range(m + 1):
        dp[i][0] = i

    # Initialize first row
    for j in range(n + 1):
        dp[0][j] = j

    # Calculate distance
    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(

                dp[i - 1][j] + 1,       # Deletion

                dp[i][j - 1] + 1,       # Insertion

                dp[i - 1][j - 1] + cost # Substitution
            )

    return dp[m][n]


# --------------------------------------------
# 8. SPELLING CORRECTION
# --------------------------------------------

def correct_word(word):

    word = clean_text(word)

    if word == "":
        return word

    # Already correct
    if word in correct_words:
        return word

    # Check known spelling error
    if word in spelling_dict:
        return spelling_dict[word]

    # Find closest word
    min_distance = float("inf")
    best_word = word

    for correct_word_value in correct_words:

        # Avoid unnecessary comparisons
        if abs(len(word) - len(correct_word_value)) > 3:
            continue

        distance = edit_distance(
            word,
            correct_word_value
        )

        if distance < min_distance:

            min_distance = distance
            best_word = correct_word_value

    return best_word


# --------------------------------------------
# 9. SEARCH QUERY CORRECTION
# --------------------------------------------

def correct_query(query):

    words = query.lower().split()

    corrected_words = []

    for word in words:

        # Remove punctuation
        cleaned_word = re.sub(
            r'[^a-zA-Z]',
            '',
            word
        )

        if cleaned_word:

            corrected = correct_word(
                cleaned_word
            )

            corrected_words.append(corrected)

    return " ".join(corrected_words)
test_words = [
    "computr",
    "recieve",
    "adress",
    "acheive",
    "freind"
]

print("\n")
print("=" * 50)
print("WORD SPELLING CORRECTION")
print("=" * 50)

for word in test_words:

    corrected = correct_word(word)

    print(
        word,
        "->",
        corrected
    )
queries = [
    "best computr",
    "recieve my adress",
    "learn machin lerning",
    "good freind",
    "acheive success"
]

print("\n")
print("=" * 50)
print("SEARCH QUERY CORRECTION")
print("=" * 50)

for query in queries:

    corrected = correct_query(query)

    print("\nOriginal Query :", query)

    print("Corrected Query:", corrected)

print("\n")
print("=" * 50)
print("SEARCH ENGINE / AUTOCOMPLETE")
print("=" * 50)

query = input("\nEnter your search query: ")

corrected_query = correct_query(query)

print("\nOriginal Query :", query)

print("Corrected Query:", corrected_query)
df.to_csv(
    "spellcheck1_cleaned.csv",
    index=False
)

print("\n")
print("Cleaned dataset saved successfully!")
print("File: spellcheck1_cleaned.csv")

Dataset loaded successfully!

First 10 rows:
   incorrect  correct
0    $Albert      NaN
1         Ab      NaN
2   $America      NaN
3    Ameraca      NaN
4    Amercia      NaN
5  $American      NaN
6   Ameracan      NaN
7     $April      NaN
8      Apirl      NaN
9  $Austrian      NaN

Columns:
Index(['incorrect', 'correct'], dtype='str')

Incorrect column: incorrect
Correct column: correct

Cleaned Dataset:
  incorrect correct
0    albert     nan
1        ab     nan
2   america     nan
3   ameraca     nan
4   amercia     nan
5  american     nan
6  ameracan     nan
7     april     nan
8     apirl     nan
9  austrian     nan

Dataset size: 42269

Spelling dictionary created.
Number of correct words: 1


WORD SPELLING CORRECTION
computr -> computr
recieve -> nan
adress -> nan
acheive -> nan
freind -> nan


SEARCH QUERY CORRECTION

Original Query : best computr
Corrected Query: nan computr

Original Query : recieve my adress
Corrected Query: nan nan nan

Original Query : learn machin ler